<a href="https://colab.research.google.com/github/jyryu3161/2026CADD/blob/main/week7_BindingSite_Prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Setting environment

In [1]:

# PDB 구조 수정 및 Docking receptor 준비
!pip -q install pdbfixer openmm meeko rdkit scipy gemmi


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 665.9/665.9 kB 16.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.4/14.4 MB 54.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 31.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.8/31.8 MB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 27.4 MB/s eta 0:00:00


In [2]:

import requests
from io import StringIO
from collections import Counter
from pdbfixer import PDBFixer
from openmm.app import PDBFile

# ==============================
# 사용자 설정
# ==============================
PDB_ID = "1M17"
CHAIN_ID = "A"      # None이면 모든 단백질 chain 유지
PH = 7.4

# ==============================
# 1. PDB 다운로드
# ==============================
url = f"https://files.rcsb.org/download/{PDB_ID.upper()}.pdb"
response = requests.get(url, timeout=60)
response.raise_for_status()

raw_pdb = f"{PDB_ID}_original.pdb"
with open(raw_pdb, "w") as f:
    f.write(response.text)

print("Downloaded:", raw_pdb)

# ==============================
# 2. 원본 Ligand 확인
# ==============================
hetero_residues = set()

for line in response.text.splitlines():
    if line.startswith("HETATM"):
        name = line[17:20].strip()
        chain = line[21].strip()
        number = line[22:27].strip()

        if name not in ["HOH", "WAT"]:
            hetero_residues.add((name, chain, number))

print("Original heterogens:", sorted(hetero_residues))

# ==============================
# 3. 구조 불러오기
# ==============================
fixer = PDBFixer(pdbfile=StringIO(response.text))

# 선택한 chain만 유지
if CHAIN_ID is not None:
    chain_indices = [
        i for i, chain in enumerate(fixer.topology.chains())
        if chain.id != CHAIN_ID
    ]

    fixer.removeChains(chain_indices)

# ==============================
# 4. 비표준 아미노산 처리
# ==============================
fixer.findNonstandardResidues()
print("Nonstandard residues:", fixer.nonstandardResidues)

fixer.replaceNonstandardResidues()

# ==============================
# 5. Ligand, water, 이온 제거
# ==============================
fixer.removeHeterogens(False)

# ==============================
# 6. Missing atoms 보완
# ==============================
fixer.findMissingResidues()

# 긴 missing loop는 자동 생성하지 않음
fixer.missingResidues = {}

fixer.findMissingAtoms()
fixer.addMissingAtoms()

# ==============================
# 7. Hydrogen 추가
# ==============================
fixer.addMissingHydrogens(PH)

# ==============================
# 8. PDB 저장
# ==============================
prepared_pdb = f"{PDB_ID}_prepared.pdb"

with open(prepared_pdb, "w") as f:
    PDBFile.writeFile(
        fixer.topology,
        fixer.positions,
        f,
        keepIds=True
    )

print("Prepared protein:", prepared_pdb)
print("Atoms:", fixer.topology.getNumAtoms())
print("Residues:", fixer.topology.getNumResidues())


Downloaded: 1M17_original.pdb
Original heterogens: [('AQ4', 'A', '999')]
Nonstandard residues: []
Prepared protein: 1M17_prepared.pdb
Atoms: 5029
Residues: 312


# Binding Site Prediction

In [ ]:
!apt-get update -y
!apt-get install -y openjdk-11-jre-headless
!java -version
!wget https://github.com/rdk/p2rank/releases/download/2.4/p2rank_2.4.tar.gz
!tar -xzf p2rank_2.4.tar.gz


Hit:1 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:2 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:3 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:4 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:5 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:6 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:9 http://archive.ubuntu.com/ubuntu jammy-updates/universe amd64 Packages [1,552 kB]
Hit:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Get:11 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 Packages [3,262 kB]
Hit:12 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:13 http://archive.ubuntu.com/ubuntu jammy-up

In [ ]:
!./p2rank_2.4/prank predict -f EXP.pdb -o output_folder_EXP # .pdb 파일 이름 수정 필요
!./p2rank_2.4/prank predict -f PDB.pdb -o output_folder_PDB # .pdb 파일 이름 수정 필요

----------------------------------------------------------------------------------------------
 P2Rank 2.4
----------------------------------------------------------------------------------------------

predicting pockets for proteins from dataset [EXP.pdb]
processing [EXP.pdb] (1/1)
predicting pockets finished in 0 hours 0 minutes 39.205 seconds
results saved to directory [/content/output_folder_EXP]

----------------------------------------------------------------------------------------------
 finished successfully in 0 hours 0 minutes 42.217 seconds
----------------------------------------------------------------------------------------------
----------------------------------------------------------------------------------------------
 P2Rank 2.4
----------------------------------------------------------------------------------------------

predicting pockets for proteins from dataset [PDB.pdb]
processing [PDB.pdb] (1/1)
predicting pockets finished in 0 hours 0 minutes 19.639 seco

In [ ]:
!tar -czf output_folder_EXP.tar.gz ./output_folder_EXP/
!tar -czf output_folder_PDB.tar.gz ./output_folder_PDB/